In [ ]:
# ==============================================================================
# CELDA DE CONFIGURACIÓN PARA GOOGLE COLAB
# Ejecuta esta celda una sola vez para instalar el entorno y el motor de IA.
# ==============================================================================

import os
import time

print("0. Instalando dependencias del sistema (zstd)...")
# Actualizamos los repositorios e instalamos zstd en modo silencioso
!apt-get update -qq
!apt-get install -y zstd -qq

print("1. Instalando librerías de Python requeridas...")
!pip install PyPDF2 matplotlib pandas -q

print("2. Instalando el motor de Ollama en el servidor Linux de Colab...")
!curl -fsSL https://ollama.com/install.sh | sh

print("3. Levantando el servidor de Ollama en segundo plano...")
# Usamos nohup para que el servidor corra invisiblemente sin bloquear la celda
os.system("nohup ollama serve > ollama_server.log 2>&1 &")

# Le damos unos segundos al servidor para arrancar por completo
time.sleep(5)

print("4. Descargando el modelo Gemma (esto tomará unos minutos)...")
# Descargando la versión de 7b (cámbialo a gemma:2b si prefieres la versión ligera)
!ollama pull gemma:7b

print("✅ ¡Entorno de Inteligencia Artificial configurado con éxito!")
print("Ya puedes ejecutar las demás celdas del proyecto.")

In [ ]:
import glob
from google.colab import files

print("FASE 1: DESCARGANDO EXPEDIENTES BASE")
# Reemplaza la URL con la liga real de tu repositorio
!git clone https://github.com/VanyaKaramazov/gdg-cdmx-hackday-2026 repo_temporal
# Movemos los PDFs a la carpeta principal y borramos la carpeta temporal
!mv repo_temporal/*.pdf ./ 2>/dev/null
!rm -rf repo_temporal
print("✅ Expedientes base de prueba cargados desde el repositorio.\n")

print("FASE 2: INGESTA MANUAL (OPCIONAL)")
print("Si deseas probar el modelo con tus propios expedientes en PDF, selecciónalos ahora.")
print("Si solo quieres usar los de prueba, presiona 'Cancelar' en el cuadro de diálogo de abajo.\n")

try:
    archivos_subidos = files.upload()
    if archivos_subidos:
        print(f"✅ Se agregaron {len(archivos_subidos)} archivos manuales con éxito.")
    else:
        print("⏩ No se subieron archivos nuevos. Continuando con la base de prueba.")
except Exception as e:
    print("⏩ Carga manual cancelada o interrumpida. Continuando con la base de prueba.")

# Verificación final antes de pasar a Gemma
total_pdfs = glob.glob("*.pdf")
print("-" * 50)
print(f"LISTO: Hay un total de {len(total_pdfs)} PDFs en la cola para ser procesados por Gemma 4.")
print("-" * 50)

In [ ]:
import sqlite3

def inicializar_bd_final():
    conexion = sqlite3.connect('clinica_local.db')
    cursor = conexion.cursor()

    cursor.execute("DROP TABLE IF EXISTS pacientes")
    cursor.execute("DROP TABLE IF EXISTS registros_triage")

    cursor.execute('''
    CREATE TABLE pacientes (
        id_paciente INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_completo TEXT,
        edad INTEGER,
        sexo TEXT
    )
    ''')

    cursor.execute('''
    CREATE TABLE registros_triage (
        id_registro INTEGER PRIMARY KEY AUTOINCREMENT,
        id_paciente INTEGER,
        pa_sistolica INTEGER,
        pa_diastolica INTEGER,
        frec_cardiaca INTEGER,
        frec_respiratoria INTEGER,
        peso REAL,
        talla REAL,
        temperatura REAL,
        motivo_consulta TEXT,
        diagnostico_principal TEXT,
        nivel_urgencia TEXT,
        justificacion_clinica TEXT,
        FOREIGN KEY(id_paciente) REFERENCES pacientes(id_paciente)
    )
    ''')

    conexion.commit()
    conexion.close()
    print("✅ Base de datos relacional lista para el prototipo final.")

inicializar_bd_final()

In [ ]:
import os
import glob
import PyPDF2
import requests
import json
import sqlite3

url_local = "http://localhost:11434/api/generate"
# Busca todos los archivos .pdf en la carpeta actual
archivos_pdf = glob.glob("*.pdf")

if not archivos_pdf: # ¡Aquí está el "not" corregido!
    print("⚠️ No se encontraron archivos PDF en la carpeta actual.")
else:
    print(f"📂 Se encontraron {len(archivos_pdf)} archivos PDF. Iniciando procesamiento por lotes...\n")

    for ruta_pdf in archivos_pdf:
        nombre_archivo = os.path.basename(ruta_pdf)
        print(f"📄 Procesando: {nombre_archivo}")

        # 1. EXTRAER TEXTO
        texto_pdf = ""
        try:
            with open(ruta_pdf, "rb") as archivo:
                lector_pdf = PyPDF2.PdfReader(archivo)
                for pagina in lector_pdf.pages:
                    texto_pdf += pagina.extract_text() + "\n"
        except Exception as e:
            print(f"   ❌ Error al leer el PDF: {e}")
            continue

        if not texto_pdf.strip():
            print("   ⚠️ El PDF está vacío o es una imagen sin OCR. Saltando...")
            continue

        # 2. INFERENCIA CON GEMMA 4 (Con el Prompt Anti-Nulos)
        prompt_maestro = f"""
        Eres un motor de inferencia clínica. Analiza este texto de un expediente médico:

        "{texto_pdf}"

        TAREA: Reemplaza las instrucciones en mayúsculas dentro del siguiente JSON con los datos reales extraídos del texto.
        REGLA ESTRICTA: Si un dato definitivamente no está en el texto, reemplaza la instrucción con la palabra null (sin comillas). Los números deben ser enteros o decimales.

        Devuelve ÚNICAMENTE el objeto JSON válido:
        {{
          "nombre_completo": "EXTRAER NOMBRE AQUÍ",
          "edad": "EXTRAER EDAD AQUÍ COMO NUMERO",
          "sexo": "EXTRAER SEXO AQUÍ",
          "pa_sistolica": "EXTRAER PRESION SISTOLICA AQUI COMO NUMERO",
          "pa_diastolica": "EXTRAER PRESION DIASTOLICA AQUI COMO NUMERO",
          "frec_cardiaca": "EXTRAER PULSO AQUI COMO NUMERO",
          "frec_respiratoria": "EXTRAER RESPIRACION AQUI COMO NUMERO",
          "peso": "EXTRAER PESO AQUI COMO NUMERO",
          "talla": "EXTRAER TALLA AQUI COMO NUMERO",
          "temperatura": "EXTRAER TEMPERATURA AQUI COMO NUMERO",
          "motivo_consulta": "RESUMIR MOTIVO AQUI",
          "diagnostico_principal": "EXTRAER DIAGNOSTICO AQUI",
          "nivel_urgencia": "EVALUAR COMO ALTA, MEDIA O BAJA",
          "justificacion_clinica": "EXPLICAR RAZONAMIENTO AQUI"
        }}
        """

        try:
            # Configuramos el payload asegurando el nombre exacto y el formato JSON
            payload = {
                "model": "gemma:7b", # ⚠️ CAMBIA ESTO A gemma:2b SI ESA FUE LA VERSIÓN QUE DESCARGASTE
                "prompt": prompt_maestro,
                "stream": False,
                "format": "json"     # ⚠️ Esto obliga al modelo a no usar texto conversacional
            }

            resp = requests.post(url_local, json=payload)

            # Pequeño seguro de depuración para ver qué dice el servidor si falla
            if resp.status_code != 200:
                print(f"   ❌ Error del servidor Ollama: {resp.text}")
                continue

            resultado_crudo = resp.json().get("response", "").strip()

            # Limpieza de markdown
            texto_json = resultado_crudo
            if texto_json.startswith("```json"): texto_json = texto_json[7:-3].strip()
            elif texto_json.startswith("```"): texto_json = texto_json[3:-3].strip()

            datos = json.loads(texto_json)

            # 3. INSERCIÓN EN SQLITE
            conexion = sqlite3.connect('clinica_local.db')
            cursor = conexion.cursor()

            # Guardar paciente
            cursor.execute('''INSERT INTO pacientes (nombre_completo, edad, sexo) VALUES (?, ?, ?)''',
                           (datos.get('nombre_completo'), datos.get('edad'), datos.get('sexo')))
            id_pac = cursor.lastrowid

            # Guardar triaje
            cursor.execute('''INSERT INTO registros_triage (id_paciente, pa_sistolica, pa_diastolica, frec_cardiaca, frec_respiratoria, peso, talla, temperatura, motivo_consulta, diagnostico_principal, nivel_urgencia, justificacion_clinica) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                           (id_pac, datos.get('pa_sistolica'), datos.get('pa_diastolica'), datos.get('frec_cardiaca'), datos.get('frec_respiratoria'), datos.get('peso'), datos.get('talla'), datos.get('temperatura'), datos.get('motivo_consulta'), datos.get('diagnostico_principal'), datos.get('nivel_urgencia'), datos.get('justificacion_clinica')))

            conexion.commit()
            conexion.close()
            print("   ✅ Extraído, analizado y guardado en SQLite con éxito.")

        except Exception as e:
            print(f"   ❌ Error estructurando JSON o guardando en BD: {e}")

    print("\n🚀 ¡Procesamiento masivo finalizado!")

In [ ]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect('clinica_local.db')
df_pacientes = pd.read_sql_query("SELECT * FROM pacientes", conexion)
df_registros = pd.read_sql_query("SELECT * FROM registros_triage", conexion)
conexion.close()

print("👤 === PACIENTES ===")
display(df_pacientes)

print("\n🏥 === INFERENCIA CLÍNICA (GEMMA 4) ===")
display(df_registros[['id_paciente', 'diagnostico_principal', 'nivel_urgencia', 'justificacion_clinica']])

In [ ]:
import sqlite3
import requests

url_local = "http://localhost:11434/api/generate"

MODELO_ACTUAL = "gemma:7b"

def consulta_estadistica_interactiva():
    print("🏥 === ASISTENTE MÉDICO IA INICIADO ===")
    print("Escribe tu consulta en lenguaje natural. Escribe 'salir' para terminar.\n")

    while True:
        pregunta_usuario = input("🔎 Tu consulta: ")

        if pregunta_usuario.lower() in ['salir', 'exit', 'quit', 'terminar']:
            print("👋 Cerrando asistente. ¡Buen trabajo!")
            break

        if not pregunta_usuario.strip():
            continue

        print(f"\n🧠 Procesando: '{pregunta_usuario}'...")

        # ==========================================
        # 1. TRADUCCIÓN A SQL
        # ==========================================
        prompt_sql = f"""
        Eres un experto en bases de datos relacionales y sintaxis SQLite.
        Tienes dos tablas conectadas por 'id_paciente':
        1. 'pacientes' (id_paciente, nombre_completo, edad, sexo)
        2. 'registros_triage' (id_registro, id_paciente, pa_sistolica, pa_diastolica, frec_cardiaca, frec_respiratoria, peso, talla, temperatura, motivo_consulta, diagnostico_principal, nivel_urgencia)

        Traduce esta pregunta del usuario a una consulta SQL válida.
        REGLA ESTRICTA: Devuelve ÚNICAMENTE el código SQL en texto plano, sin formato markdown, sin comillas invertidas y sin explicaciones.

        Pregunta: "{pregunta_usuario}"
        """

        try:
            resp_sql = requests.post(url_local, json={"model": MODELO_ACTUAL, "prompt": prompt_sql, "stream": False})

            if resp_sql.status_code != 200:
                print(f"❌ Error de Ollama al generar SQL: {resp_sql.text}")
                continue

            query_sql = resp_sql.json().get("response", "").strip()

            # Limpieza exhaustiva de markdown (por si Gemma intenta decorar el código)
            if query_sql.startswith("```sql"): query_sql = query_sql[6:-3].strip()
            elif query_sql.startswith("```"): query_sql = query_sql[3:-3].strip()

            print(f"💻 Query SQL Generada: {query_sql}")

            # ==========================================
            # 2. EJECUCIÓN EN SQLITE
            # ==========================================
            conexion = sqlite3.connect('clinica_local.db')
            cursor = conexion.cursor()
            cursor.execute(query_sql)
            resultados_db = cursor.fetchall()
            conexion.close()

            print(f"📊 Datos crudos extraídos de BD: {resultados_db}")

        except Exception as e:
            print(f"❌ Error al ejecutar la consulta en la base de datos: {e}\n")
            continue

        # ==========================================
        # 3. TRADUCCIÓN A LENGUAJE NATURAL
        # ==========================================
        prompt_humano = f"""
        Eres un analista de datos clínicos.
        Pregunta del usuario: "{pregunta_usuario}"
        Datos obtenidos de la BD: {resultados_db}

        Responde a la pregunta basándote ÚNICAMENTE en los datos obtenidos.
        Sé directo, claro y profesional. No menciones la consulta SQL en tu respuesta. Si los datos están vacíos (una lista vacía []), indica amablemente que no hay registros que coincidan.
        """

        try:
            resp_humano = requests.post(url_local, json={"model": MODELO_ACTUAL, "prompt": prompt_humano, "stream": False})
            respuesta_final = resp_humano.json().get("response", "").strip()

            print("\n✨ RESPUESTA:")
            print("-" * 50)
            print(respuesta_final)
            print("-" * 50 + "\n")

        except Exception as e:
            print(f"❌ Error al generar la respuesta en lenguaje natural: {e}\n")

# Ejecutar el asistente
consulta_estadistica_interactiva()